# DSA-01 · Binary search

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/DSA-01.ipynb)

**Code:** [kotlin-dsa-kata/src/main/kotlin/patterns/binarysearch](https://github.com/BuiltBySwap/kotlin-dsa-kata/tree/main/src/main/kotlin/patterns/binarysearch)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: DSA-01
title: "Binary search"
track: dsa
code: "kotlin-dsa-kata/src/main/kotlin/patterns/binarysearch"
date: 2026-10-05
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "With the closed range, everything left of `left` is smaller and everything right of `right` is larger, so after the loop `left` is the insert position."
tags: [DSA-01, dsa, binary-search, kotlin]
pending:
  - "Write the invariant sentence in the LC 704 file"
  - "The `<` experiment: which 3 tests fail and why"
  - "Half-open searchInsertFirst with the duplicates test"
  - "One line: when closed [left, right] vs half-open [lo, hi)"
  - "Redo LC 560 from a blank file (Sun 11 Oct)"
```

## Goal
Get the boundaries right every time by thinking in invariants.

## What I learned (plain words)
| | Closed `[left, right]` | Half-open `[lo, hi)` |
|---|---|---|
| Start | `right = n - 1` | `hi = n` |
| Loop | `while (left <= right)` | `while (lo < hi)` |
| Move | `left = mid + 1`, `right = mid - 1` | `lo = mid + 1`, `hi = mid` |
| Best for | Exact match (return `mid`) | Boundaries: first index where a condition holds |

- Compute the middle as `left + (right - left) / 2` to avoid `Int` overflow.
- Classic bugs: `right = n` with `<=` (out of bounds), `<` with `n - 1` (misses the last candidate), `left = mid` with `<=` (infinite loop).
- For LC 35, `return left` works with the closed loop because values are distinct; with duplicates, early `return mid` breaks "first index ≥ target".

## Examples (run them)

Python-kernel notebook, Kotlin cells: each `%%kotlin` cell is a script that runs top to bottom. Put the function and its checks in the same cell. The first Kotlin cell takes about a minute (it downloads the compiler); later cells take about 8 seconds.

In [ ]:
# Setup: run once per Colab session. It adds the %%kotlin cell magic.
!curl -sL -o kotlin_magic.py https://raw.githubusercontent.com/BuiltBySwap/engineering-lab/main/tools/kotlin_magic.py
%run kotlin_magic.py

### LC 704: exact match, closed range `[left, right]`

In [ ]:
%%kotlin
import kotlin.test.assertEquals

// LC 704: exact match, closed range [left, right]
fun search(nums: IntArray, target: Int): Int {
    var left = 0
    var right = nums.size - 1
    while (left <= right) {
        val mid = left + (right - left) / 2
        when {
            nums[mid] == target -> return mid
            nums[mid] < target -> left = mid + 1
            else -> right = mid - 1
        }
    }
    return -1
}

val sample = intArrayOf(-1, 0, 3, 5, 9, 12)
assertEquals(4, search(sample, 9))
assertEquals(-1, search(sample, 2))
assertEquals(0, search(sample, -1))
assertEquals(5, search(sample, 12))
assertEquals(-1, search(sample, -5))
assertEquals(-1, search(sample, 100))
assertEquals(0, search(intArrayOf(5), 5))
assertEquals(-1, search(intArrayOf(5), 3))
assertEquals(-1, search(intArrayOf(), 1))
assertEquals(0, search(intArrayOf(1, 3), 1))
assertEquals(1, search(intArrayOf(1, 3), 3))
println("LC 704: all 11 checks passed")

### LC 35: the same loop, return `left`, and the duplicates caveat

In [ ]:
%%kotlin
import kotlin.test.assertEquals

// LC 35: the same loop, but return `left` when the target is missing
fun searchInsert(nums: IntArray, target: Int): Int {
    var left = 0
    var right = nums.size - 1
    while (left <= right) {
        val mid = left + (right - left) / 2
        when {
            nums[mid] == target -> return mid
            nums[mid] < target -> left = mid + 1
            else -> right = mid - 1
        }
    }
    return left
}

val sorted = intArrayOf(1, 3, 5, 6)
assertEquals(2, searchInsert(sorted, 5))
assertEquals(1, searchInsert(sorted, 2))
assertEquals(4, searchInsert(sorted, 7))
assertEquals(0, searchInsert(sorted, 0))
assertEquals(0, searchInsert(intArrayOf(1), 1))
assertEquals(1, searchInsert(intArrayOf(1), 2))
assertEquals(0, searchInsert(intArrayOf(), 5))
println("LC 35: all 7 checks passed (the values are distinct)")

// The caveat: with duplicates, returning `mid` early is NOT the first index >= target
val withDuplicates = intArrayOf(1, 3, 3, 3, 5)
println("closed version says ${searchInsert(withDuplicates, 3)}; the first index with a value >= 3 is 1")

### Experiment: change `<=` to `<`

**Predict first:** which checks will fail, and what do they have in common?

In [ ]:
%%kotlin
// The experiment: change <= to < and see which checks fail
fun searchStrict(nums: IntArray, target: Int): Int {
    var left = 0
    var right = nums.size - 1
    while (left < right) {            // <- the only change
        val mid = left + (right - left) / 2
        when {
            nums[mid] == target -> return mid
            nums[mid] < target -> left = mid + 1
            else -> right = mid - 1
        }
    }
    return -1
}

val sample = intArrayOf(-1, 0, 3, 5, 9, 12)
val cases = listOf(
    Triple("found", sample to 9, 4),
    Triple("notFound", sample to 2, -1),
    Triple("firstElement", sample to -1, 0),
    Triple("lastElement", sample to 12, 5),
    Triple("smallerThanAll", sample to -5, -1),
    Triple("biggerThanAll", sample to 100, -1),
    Triple("singleHit", intArrayOf(5) to 5, 0),
    Triple("singleMiss", intArrayOf(5) to 3, -1),
    Triple("empty", intArrayOf() to 1, -1),
    Triple("two, target 1", intArrayOf(1, 3) to 1, 0),
    Triple("two, target 3", intArrayOf(1, 3) to 3, 1),
)
for ((name, input, expected) in cases) {
    val actual = searchStrict(input.first, input.second)
    println("${if (actual == expected) "PASS" else "FAIL"}  $name: expected $expected, got $actual")
}

### Exercise: the half-open version

Replace the `TODO` line. The checks stay red until it is right. The last check (duplicates) is the one the closed version fails.

In [ ]:
%%kotlin
import kotlin.test.assertEquals

// Exercise: half-open range [lo, hi). Return the FIRST index whose value is >= target.
fun searchInsertFirst(nums: IntArray, target: Int): Int {
    var lo = 0
    var hi = nums.size
    while (lo < hi) {
        val mid = lo + (hi - lo) / 2
        TODO("move lo or hi, then delete this line")
    }
    return lo
}

assertEquals(2, searchInsertFirst(intArrayOf(1, 3, 5, 6), 5))
assertEquals(1, searchInsertFirst(intArrayOf(1, 3, 5, 6), 2))
assertEquals(4, searchInsertFirst(intArrayOf(1, 3, 5, 6), 7))
assertEquals(0, searchInsertFirst(intArrayOf(), 5))
assertEquals(1, searchInsertFirst(intArrayOf(1, 3, 3, 3, 5), 3))   // duplicates: the closed version fails this
println("half-open version: all checks passed")

## Bugs and fixes
| Error or symptom | Cause | Fix |
|---|---|---|
| Test file in `src/main` failed to compile | `kotlin.test` is only available to tests | Move it to `src/test` |

## In my own words
✏️ The invariant sentence for the closed template.